In [27]:
import os
import argparse
import logging
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

In [28]:
# set project
project_name = "1329"

In [29]:
# set output project directory
output_project_dir = os.path.join("/scratch/jh2589/lasl_analysis/", project_name)
# load the project geojson
project_geojson_file = os.path.join("/maps/mwd24/tmf-data/projects/", f"{project_name}.geojson")
project_gdf = gpd.read_file(project_geojson_file)

In [30]:
# load all_k_grids.parquet in k_grids directory
k_grids_dir = os.path.join(output_project_dir, "k_grids")
k_grids_file = os.path.join(k_grids_dir, "all_k_grids.parquet")
k_grids_df = pd.read_parquet(k_grids_file)

In [31]:
# split into pixels/rows that fall within the project boundary and those that do not
# based on the "lat" "lng" cols
k_grids_gdf = gpd.GeoDataFrame(k_grids_df, geometry=gpd.points_from_xy(k_grids_df["lng"], k_grids_df["lat"]), crs="EPSG:4326")

In [33]:
# split into points that fall within the project boundary and those that do not (vectorised, faster)
# ensure matching CRS
if project_gdf.crs != k_grids_gdf.crs:
    project_gdf = project_gdf.to_crs(k_grids_gdf.crs)

# fast spatial join
joined = gpd.sjoin(k_grids_gdf, project_gdf[['geometry']], how='left', predicate='within')

within_project_gdf = joined[joined['index_right'].notna()].drop(columns=['index_right'])
buffer_project_gdf = joined[joined['index_right'].isna()].drop(columns=['index_right'])


In [34]:
print(f"Number of pixels within project boundary: {len(within_project_gdf)}"
      f"\nNumber of pixels buffer project boundary: {len(buffer_project_gdf)}")

Number of pixels within project boundary: 332892
Number of pixels buffer project boundary: 1231853


In [35]:
def luc_summary(df, prefix="luc_", luc_class=1, pixel_area_m2=900):
    """
    Return DataFrame indexed by luc column with columns: count, proportion (0-1), area_ha.
    """
    luc_cols = [c for c in df.columns if c.startswith(prefix)]
    rows = []
    total = len(df)
    for col in luc_cols:
        count = int((df[col] == luc_class).sum())
        proportion = (count / total) if total else 0.0
        area_ha = count * pixel_area_m2 / 10000.0
        rows.append({"col": col, "count": count, "proportion": proportion, "area_ha": area_ha})
    return pd.DataFrame(rows).set_index("col").sort_index()


In [36]:
undisturbed_summary_proj_df = luc_summary(within_project_gdf, prefix="luc_")
undisturbed_summary_buffer_proj_df = luc_summary(buffer_project_gdf, prefix="luc_")

In [37]:
# load pairs paruqets in pairs_1000 that don't end in _matchless
parquets_dir = os.path.join(output_project_dir, "pairs_1000")
parquet_files = [f for f in os.listdir(parquets_dir) if f.endswith(".parquet") and not f.endswith("_matchless.parquet")]
pairs_dfs = [pd.read_parquet(os.path.join(parquets_dir, f)) for f in parquet_files]
# concatenate all pairs dataframes
pairs_df = pd.concat(pairs_dfs, ignore_index=True)

In [38]:
# split pairs into those that fall within the project boundary and those that do not
pairs_gdf = gpd.GeoDataFrame(pairs_df, geometry=gpd.points_from_xy(pairs_df["k_lng"], pairs_df["k_lat"]), crs="EPSG:4326")
joined_pairs = gpd.sjoin(pairs_gdf, project_gdf[['geometry']], how='left', predicate='within')
within_project_pairs_gdf = joined_pairs[joined_pairs['index_right'].notna()].drop(columns=['index_right'])
buffer_project_pairs_gdf = joined_pairs[joined_pairs['index_right'].isna()].drop(columns=['index_right'])

In [39]:
print(f"Number of pixels within project boundary: {len(within_project_gdf)}")
print(f"Number of pixels outside project boundary: {len(outside_project_gdf)}")

Number of pixels within project boundary: 332892
Number of pixels outside project boundary: 1231853


In [40]:
contrl_undisturbed_summary_proj_df = luc_summary(within_project_pairs_gdf, prefix="s_luc_")
contrl_undisturbed_summary_buffer_proj_df = luc_summary(buffer_project_pairs_gdf, prefix="s_luc_")

In [41]:
# extract proportion of undisturbed controls.loc["s_luc_1", "proportion"]

KeyError: 's_luc_1'